In [1]:
# Check the concentration maps:

# Load them, apply mask
# compute mean/median (plot hists)
# and compare with literature values


import inputs
import file
import tools
import numpy as np

In [2]:
target_resolution_xyz = (112, 128, 80)
target_gpu_smaller_tasks = 0

In [3]:
config_file_name = "paths_09012026.json"
configurator = inputs.Configurator(path_folder="../config/",
                                 file_name=config_file_name)
configurator.load()

# Load and interpolate the Mask

In [4]:
metabolic_mask = file.Mask(configurator=configurator)
metabolic_mask.load(mask_name="metabolites", data_type=np.float32)
metabolic_mask.mask.data = tools.InterpolationTools().interpolate(array=metabolic_mask.mask.data, target_size=(440, 440, 266), order=0, compute_on_device="gpu", return_on_device="cpu", target_gpu=target_gpu_smaller_tasks)

fraction_GM = file.ParameterMaps(configurator, map_type_name="segmentation").load(map_key="segmentations.GM_tissue_fraction")
fraction_WM = file.ParameterMaps(configurator, map_type_name="segmentation").load(map_key="segmentations.WM_tissue_fraction")
fraction_CSF = file.ParameterMaps(configurator, map_type_name="segmentation").load(map_key="segmentations.CSF_tissue_fraction")


fraction_GM_data = fraction_GM.loaded_maps["segmentation"]
fraction_WM_data = fraction_WM.loaded_maps["segmentation"]
fraction_CSF_data = fraction_CSF.loaded_maps["segmentation"]

/tmp/ipykernel_1145/3970662806.py:1: DeprecationWarning: Mask is deprecated: Use ParameterMaps instead.
  metabolic_mask = file.Mask(configurator=configurator)


[  1  ][ success ] ---v 
                      Loaded file 'Vol_004_mask_slab.nii':
                          Shape             -> (112, 128, 80)
                          Pixel dimensions: -> (1.7188, 1.7188, 1.72)
                          Values range:     -> [0.0, 1.0]
                          Data type:        -> <class 'numpy.float32'>
                          In memory cache?  -> True
[  2  ][ success ] >> NaNs are NOT present!
[  3  ][ success ] >> Thus, loaded the 'metabolites' mask.
[  4  ][  info   ] >> Interpolate on GPU: 0
[  5  ][ success ] ---v  (automatic line breaks)
                      Interpolated on gpu and returned on cpu
                      Shape (112, 128, 80) (4.375 mebibyte) => (440, 440, 266) (196.44775390625
                      mebibyte)
[  6  ][  error  ] >> Could not convert loaded unit 'unknown' to pint unit. Therefore, assigned 'dimensionless'!
[  7  ][ success ] ---v 
                      Loaded 1 volume(s)!
                        names: ['Segm

# Load and interpolate the concentration maps

In [5]:
working_name_and_file_name = {"Glu": "MetMap_Glu_con_map_TargetRes_HiRes.nii",
                              "Gln": "MetMap_Gln_con_map_TargetRes_HiRes.nii",
                              "m-Ins": "MetMap_Ins_con_map_TargetRes_HiRes.nii",
                              "NAA": "MetMap_NAA_con_map_TargetRes_HiRes.nii",
                              "Cr+PCr": "MetMap_Cr+PCr_con_map_TargetRes_HiRes.nii",
                              "GPC+PCh": "MetMap_GPC+PCh_con_map_TargetRes_HiRes.nii",
                             }

loaded_concentration_maps = file.ParameterMaps(configurator=configurator, map_type_name="metabolites_concentration")
loaded_concentration_maps.load(
    map_key="maps.metabolites_concentration",
    #compounds=["Glu", "Gln", "m-Ins", "NAA"], <- only if subset needed! TODO: Really need this?!
    working_name_and_file_name=working_name_and_file_name,
    data_precision=32,
    verbose=False,
)

In [6]:
# Convert the file.Maps object to a ParamaterVolume of shape (metabolite, X, Y, Z) & change to float32
concentration_volume = loaded_concentration_maps.to_working(data_type="float32")

# Interpolate to the shape of the 
#concentration_volume.interpolate_maps(target_size=metabolic_mask.mask.shape, order=0, compute_on_device="gpu", target_gpu=target_gpu_smaller_tasks, verbose=False)

[ 15  ][ success ] ---v  (automatic line breaks)
                      Created stacked volume of the following maps: ['Glu', 'Gln', 'm-Ins', 'NAA',
                      'Cr+PCr', 'GPC+PCh'].
                       Map type: ....... metabolites_concentration
                       Unit: ........... millimole / liter
                       Shape: .......... (6, 440, 440, 266)
                       Data type: ...... float32
                       Space: .......... 1178.7 mebibyte
[ 16  ][ success ] >> Changed the data type from float32 (1178.7 mebibyte) -> float32 (1178.7 mebibyte)


In [7]:
concentration_volume.display_jupyter()

In [8]:
for m in concentration_volume.maps:
    print(f"{m.map_name:.<10}: mean: {np.mean(m.values[metabolic_mask.mask.data == 1])} | median: {np.median(m.values[metabolic_mask.mask.data == 1])}")

Glu.......: mean: 3.1102185249328613 | median: 0.0
Gln.......: mean: 1.1677321195602417 | median: 0.0
m-Ins.....: mean: 2.588968515396118 | median: 0.0
NAA.......: mean: 4.168789386749268 | median: 0.0
Cr+PCr....: mean: 3.0530405044555664 | median: 0.0
GPC+PCh...: mean: 0.868469774723053 | median: 0.0


In [9]:
import matplotlib.pyplot as plt
import scipy.stats as stats

fig, ax = plt.subplots(2, 6, figsize=(15, 6))

for i, m in enumerate(concentration_volume.maps):
    selected = m.values[metabolic_mask.mask.data == 1]
    selected = selected[selected > 0]

    mean_ = np.mean(selected)
    median_ = np.median(selected)

    # top row: histogram
    ax[0, i].hist(selected, int(np.sqrt(len(selected.ravel()))))
    ax[0, i].axvline(mean_, color='red')
    ax[0, i].axvline(median_, color='green')

    # bottom row: Q-Q plot
    stats.probplot(selected, dist="norm", plot=ax[1, i])

    # Shapiro-Wilk normality test
    stat, p = stats.shapiro(selected)
    normal = "normal" if p > 0.05 else "not normal"

    print(f"min: {np.min(selected):.3f}; max: {np.max(selected):.3f}; "
          f"mean: {mean_:.3f}; median: {median_:.3f}; "
          f"Shapiro W: {stat:.3f}, p: {p:.4f} ({normal})")

plt.tight_layout()
plt.show()

/opt/conda/envs/MRSI_simulation/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:586: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 5782065.
  res = hypotest_fun_out(*samples, **kwds)


min: 0.000; max: 22.662; mean: 7.428; median: 7.345; Shapiro W: 0.996, p: 0.0000 (not normal)


/opt/conda/envs/MRSI_simulation/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:586: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 5761034.
  res = hypotest_fun_out(*samples, **kwds)


min: 0.000; max: 12.570; mean: 2.799; median: 2.684; Shapiro W: 0.942, p: 0.0000 (not normal)


/opt/conda/envs/MRSI_simulation/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:586: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 5790318.
  res = hypotest_fun_out(*samples, **kwds)


...

/opt/conda/envs/MRSI_simulation/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:586: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 5789216.
  res = hypotest_fun_out(*samples, **kwds)


min: 0.000; max: 25.555; mean: 9.944; median: 10.200; Shapiro W: 0.954, p: 0.0000 (not normal)


/opt/conda/envs/MRSI_simulation/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:586: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 5790332.
  res = hypotest_fun_out(*samples, **kwds)


min: 0.000; max: 16.515; mean: 7.281; median: 7.423; Shapiro W: 0.970, p: 0.0000 (not normal)
min: 0.000; max: 5.275; mean: 2.071; median: 2.058; Shapiro W: 0.981, p: 0.0000 (not normal)


<Figure size 640x480 with 0 Axes>

In [10]:
# TODO: Check for GM, WM, CSF!!!!

# Now get literature values

In [11]:
from inputs import JsonResources

In [12]:
database = JsonResources(path_folder="../resources", file_name="chemical_compounds.json")

In [13]:
database.load()

In [14]:
print(database.get_data(key="metabolites.NAA.concentration.deGraaf.GM"))
print(database.get_data(key="metabolites.NAA.concentration.deGraaf.WM"))

print(database.get_data(key="metabolites.Cr+PCr.concentration.deGraaf.WM"))
print(database.get_data(key="metabolites.Cr+PCr.concentration.deGraaf.GM"))

{'range': [8, 10], 'unit': 'mM'}
{'range': [6, 9], 'unit': 'mM'}
{'range': [5.2, 5.7], 'unit': 'mM'}
{'range': [6.4, 9.7], 'unit': 'mM'}


In [15]:
NAA_GM = database.get_data(key="metabolites.NAA.concentration.Pouwels.GM")
print(NAA_GM)
NAA_WM = database.get_data(key="metabolites.NAA.concentration.Pouwels.WM")
print(NAA_WM)

Glu_GM = database.get_data(key="metabolites.Glu.concentration.Pouwels.GM")
print(Glu_GM)
Glu_WM = database.get_data(key="metabolites.Glu.concentration.Pouwels.WM")
print(Glu_WM)

Gln_GM = database.get_data(key="metabolites.Gln.concentration.Pouwels.GM")
print(Gln_GM)
Gln_WM = database.get_data(key="metabolites.Gln.concentration.Pouwels.WM")
print(Gln_WM)

mIns_GM = database.get_data(key="metabolites.m-Ins.concentration.Pouwels.GM")
print(mIns_GM)
mIns_WM = database.get_data(key="metabolites.m-Ins.concentration.Pouwels.WM")
print(mIns_WM)

CrPCr_GM = database.get_data(key="metabolites.Cr+PCr.concentration.Pouwels.GM")
print(CrPCr_GM)
CrPCr_WM = database.get_data(key="metabolites.Cr+PCr.concentration.Pouwels.WM")
print(CrPCr_WM)

GPCPCh_GM = database.get_data(key="metabolites.GPC+PCh.concentration.Pouwels.GM")
print(GPCPCh_GM)
GPCPCh_WM = database.get_data(key="metabolites.GPC+PCh.concentration.Pouwels.WM")
print(GPCPCh_WM)

{'frontal': {'mean': 7.7, 'sd': 1.0, 'unit': 'mM'}, 'parietal': {'mean': 8.2, 'sd': 0.8, 'unit': 'mM'}, 'occipital': {'mean': 9.2, 'sd': 0.9, 'unit': 'mM'}, 'insular': {'mean': 8.7, 'sd': 0.8, 'unit': 'mM'}}
{'frontal': {'mean': 8.1, 'sd': 0.9, 'unit': 'mM'}, 'parietal': {'mean': 8.0, 'sd': 1.0, 'unit': 'mM'}, 'occipital': {'mean': 7.8, 'sd': 0.9, 'unit': 'mM'}}
{'frontal': {'mean': 8.5, 'sd': 1.0, 'unit': 'mM'}, 'parietal': {'mean': 8.2, 'sd': 1.1, 'unit': 'mM'}, 'occipital': {'mean': 8.6, 'sd': 1.1, 'unit': 'mM'}, 'insular': {'mean': 8.8, 'sd': 0.8, 'unit': 'mM'}}
{'frontal': {'mean': 7.0, 'sd': 2.6, 'unit': 'mM'}, 'parietal': {'mean': 6.7, 'sd': 1.8, 'unit': 'mM'}, 'occipital': {'mean': 6.0, 'sd': 1.2, 'unit': 'mM'}}
{'frontal': {'mean': 4.4, 'sd': 1.4, 'unit': 'mM'}, 'parietal': {'mean': 3.8, 'sd': 1.4, 'unit': 'mM'}, 'occipital': {'mean': 3.9, 'sd': 1.1, 'unit': 'mM'}, 'insular': {'mean': 4.9, 'sd': 1.6, 'unit': 'mM'}}
{'frontal': {'mean': 1.8, 'sd': 1.6, 'unit': 'mM'}, 'parietal'

In [16]:
NAA_GM_mean    = sum(v['mean'] for v in NAA_GM.values())    / len(NAA_GM)
NAA_WM_mean    = sum(v['mean'] for v in NAA_WM.values())    / len(NAA_WM)
Glu_GM_mean    = sum(v['mean'] for v in Glu_GM.values())    / len(Glu_GM)
Glu_WM_mean    = sum(v['mean'] for v in Glu_WM.values())    / len(Glu_WM)
Gln_GM_mean    = sum(v['mean'] for v in Gln_GM.values())    / len(Gln_GM)
Gln_WM_mean    = sum(v['mean'] for v in Gln_WM.values())    / len(Gln_WM)
mIns_GM_mean   = sum(v['mean'] for v in mIns_GM.values())   / len(mIns_GM)
mIns_WM_mean   = sum(v['mean'] for v in mIns_WM.values())   / len(mIns_WM)
CrPCr_GM_mean  = sum(v['mean'] for v in CrPCr_GM.values())  / len(CrPCr_GM)
CrPCr_WM_mean  = sum(v['mean'] for v in CrPCr_WM.values())  / len(CrPCr_WM)
GPCPCh_GM_mean = sum(v['mean'] for v in GPCPCh_GM.values()) / len(GPCPCh_GM)
GPCPCh_WM_mean = sum(v['mean'] for v in GPCPCh_WM.values()) / len(GPCPCh_WM)

In [17]:
vals = map[GM_maske]
lo, hi = np.percentile(vals, [1, 99])
faktor = lit_mean_GM / vals[(vals>=lo)&(vals<=hi)].mean()

NameError: name 'GM_maske' is not defined

In [ ]:
# (Literaturwerte und Gewichtung) Hier berechne nun die means der jeweiligen GM und WM fraction des Gehirns

# GM
mean_NAA_GM_map    = (NAA_GM_mean    * metabolic_mask.mask.data * fraction_GM_data).sum() / (metabolic_mask.mask.data * fraction_GM_data).sum()
mean_Glu_GM_map    = (Glu_GM_mean    * metabolic_mask.mask.data * fraction_GM_data).sum() / (metabolic_mask.mask.data * fraction_GM_data).sum()
mean_Gln_GM_map    = (Gln_GM_mean    * metabolic_mask.mask.data * fraction_GM_data).sum() / (metabolic_mask.mask.data * fraction_GM_data).sum()
mean_mIns_GM_map   = (mIns_GM_mean   * metabolic_mask.mask.data * fraction_GM_data).sum() / (metabolic_mask.mask.data * fraction_GM_data).sum()
mean_CrPCr_GM_map  = (CrPCr_GM_mean  * metabolic_mask.mask.data * fraction_GM_data).sum() / (metabolic_mask.mask.data * fraction_GM_data).sum()
mean_GPCPCh_GM_map = (GPCPCh_GM_mean * metabolic_mask.mask.data * fraction_GM_data).sum() / (metabolic_mask.mask.data * fraction_GM_data).sum()

# WM
mean_NAA_WM_map    = (NAA_WM_mean    * metabolic_mask.mask.data * fraction_WM_data).sum() / (metabolic_mask.mask.data * fraction_WM_data).sum()
mean_Glu_WM_map    = (Glu_WM_mean    * metabolic_mask.mask.data * fraction_WM_data).sum() / (metabolic_mask.mask.data * fraction_WM_data).sum()
mean_Gln_WM_map    = (Gln_WM_mean    * metabolic_mask.mask.data * fraction_WM_data).sum() / (metabolic_mask.mask.data * fraction_WM_data).sum()
mean_mIns_WM_map   = (mIns_WM_mean   * metabolic_mask.mask.data * fraction_WM_data).sum() / (metabolic_mask.mask.data * fraction_WM_data).sum()
mean_CrPCr_WM_map  = (CrPCr_WM_mean  * metabolic_mask.mask.data * fraction_WM_data).sum() / (metabolic_mask.mask.data * fraction_WM_data).sum()
mean_GPCPCh_WM_map = (GPCPCh_WM_mean * metabolic_mask.mask.data * fraction_WM_data).sum() / (metabolic_mask.mask.data * fraction_WM_data).sum()

In [ ]:
#

In [ ]:
fraction_GM_data.shape

In [ ]:
metabolic_mask.mask.data.shape

In [ ]:
import numpy as np

lit_GM = {
    'NAA': NAA_GM_mean, 'Glu': Glu_GM_mean, 'Gln': Gln_GM_mean,
    'm-Ins': mIns_GM_mean, 'Cr+PCr': CrPCr_GM_mean, 'GPC+PCh': GPCPCh_GM_mean,
}  # Keys an m.map_name anpassen!

w_GM = metabolic_mask.mask.data * fraction_GM_data

for m in concentration_volume.maps:
    vals = m.values                      # gemessene, dimensionslose Map
    sel = w_GM > 0
    lo, hi = np.percentile(vals[sel], [1, 99])      # robust: 1–99 Perzentil
    w_clip = w_GM * ((vals >= lo) & (vals <= hi))

    map_mean_GM = (vals * w_clip).sum() / w_clip.sum()   # gewichteter Map-Mean
    faktor_GM = lit_GM[m.map_name] / map_mean_GM         # Skalierungsfaktor

    print(f"{m.map_name:.<10}: map_mean={map_mean_GM:.4g}  faktor={faktor_GM:.4g}")

In [ ]:
import numpy as np

# (Literature values) The Pouwels-averaged reference means per compartment (in mM)
lit_GM = {
    'NAA': NAA_GM_mean, 'Glu': Glu_GM_mean, 'Gln': Gln_GM_mean,
    'm-Ins': mIns_GM_mean, 'Cr+PCr': CrPCr_GM_mean, 'GPC+PCh': GPCPCh_GM_mean,
}
lit_WM = {
    'NAA': NAA_WM_mean, 'Glu': Glu_WM_mean, 'Gln': Gln_WM_mean,
    'm-Ins': mIns_WM_mean, 'Cr+PCr': CrPCr_WM_mean, 'GPC+PCh': GPCPCh_WM_mean,
}

# (Tissue weights) Partial-volume weights, only GM and WM (CSF excluded, metabolite-free)
# fraction maps are in range 0-1 and GM+WM+CSF sum to 1 per voxel
w_GM = metabolic_mask.mask.data * fraction_GM_data
w_WM = metabolic_mask.mask.data * fraction_WM_data

# (Containers) Store the weighted map means and the resulting scaling factors per metabolite
map_mean_GM, map_mean_WM = {}, {}
faktor_GM, faktor_WM = {}, {}

for m in concentration_volume.maps:
    vals = m.values                                  # the measured, dimensionless map

    # --- GM ---
    # (Robustness) Determine 1-99 percentile bounds over the GM-weighted voxels only,
    # so artifact outliers (hot voxels, partial-volume edges) cannot drag the anchor mean
    sel_GM = w_GM > 0
    lo, hi = np.percentile(vals[sel_GM], [1, 99])
    # (Clip) Zero the weights of outlier voxels (we only exclude them from the anchor, not the data)
    w_GM_clip = w_GM * ((vals >= lo) & (vals <= hi))
    # (Weighted mean) Tissue-weighted mean of the measured map: sum(value*weight) / sum(weight)
    map_mean_GM[m.map_name] = (vals * w_GM_clip).sum() / w_GM_clip.sum()
    # (Factor) Anchor the measured map mean to the literature GM mean
    faktor_GM[m.map_name] = lit_GM[m.map_name] / map_mean_GM[m.map_name]

    # --- WM ---
    # (Robustness) Same 1-99 percentile clipping, now over the WM-weighted voxels
    sel_WM = w_WM > 0
    lo, hi = np.percentile(vals[sel_WM], [1, 99])
    w_WM_clip = w_WM * ((vals >= lo) & (vals <= hi))
    map_mean_WM[m.map_name] = (vals * w_WM_clip).sum() / w_WM_clip.sum()
    faktor_WM[m.map_name] = lit_WM[m.map_name] / map_mean_WM[m.map_name]

    # (Report) Show both compartment factors side by side as a sanity check
    print(f"{m.map_name:.<10}: "
          f"GM map_mean={map_mean_GM[m.map_name]:.4g} faktor={faktor_GM[m.map_name]:.4g} | "
          f"WM map_mean={map_mean_WM[m.map_name]:.4g} faktor={faktor_WM[m.map_name]:.4g}")